In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Ініціалізація кубітів

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Коли схема виконується на квантовому процесорі (QPU) IBM&reg;, на початку схеми зазвичай вставляється неявне скидання, щоб гарантувати ініціалізацію кубітів до нуля. Це контролюється прапором `init_qubits`, встановленим як [параметр виконання примітиву](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Однак процес скидання не є ідеальним, що призводить до помилок підготовки стану. Щоб пом'якшити помилку, система також вставляє час затримки повторення (`rep_delay`) між схемами. Кожен бекенд має різне значення `rep_delay` за замовчуванням, але воно зазвичай більше за T1, щоб середовище встигло скинути кубіти. Значення `rep_delay` за замовчуванням можна отримати, виконавши `backend.default_rep_delay`.

Усі QPU IBM використовують виконання з динамічною частотою повторення, що дозволяє змінювати `rep_delay` для кожного завдання. Схеми, які ти подаєш у завданні з примітивом, групуються разом для виконання на QPU. Ці схеми виконуються шляхом перебору схем для кожного запитаного знімка; виконання відбувається по стовпцях у матриці схем і знімків, як показано на наступному малюнку.

![Перший стовпець представляє знімок 0. Схеми виконуються по порядку від 0 до 3. Другий стовпець представляє знімок 1. Схеми виконуються по порядку від 0 до 3. Решта стовпців слідують тому ж шаблону.](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Матриця виконання по стовпцях")

Оскільки `rep_delay` вставляється між схемами, кожен знімок під час виконання зустрічає цю затримку. Тому зменшення `rep_delay` скорочує загальний час виконання на QPU, але коштом збільшення частоти помилок підготовки стану, як видно на наступному зображенні:

![Це зображення показує, що зі зменшенням значення `rep_delay` частота помилок підготовки стану зростає.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Затримка повторення проти частоти помилок")

Встановлення одночасно `rep_delay=0` та `init_qubits=False` фактично «зливає» схеми разом, оскільки кубіти починатимуться у кінцевому стані від попереднього знімка.

Зауваж, що хоча схеми в завданні з примітивом групуються разом для виконання на QPU, порядок виконання схем з PUB не гарантується. Тому, навіть якщо ти подаєш `pubs=[pub1, pub2]`, немає гарантії, що схеми з `pub1` будуть виконані раніше, ніж схеми з `pub2`. Також немає гарантії, що схеми з одного завдання будуть виконані як один пакет на QPU.

## Вказати rep_delay для завдання з примітивом

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Наступні кроки
> **Tip:** - Спробуй приклад у посібнику [Квантовий алгоритм апроксимаційної оптимізації (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Ознайомся з тим, як [почати роботу з примітивами.](./get-started-with-primitives)